# FocusLamp – Dashboard Streamlit (dijalankan dari Google Colab)

Notebook ini menyalakan **web dashboard FocusLamp** berbasis Streamlit di Google Colab, lalu memberi **link** yang bisa dibuka di browser laptop atau HP. Setelah web menyala, semua pengaturan dilakukan dari halaman web, bukan dengan mengetik perintah di sel: atur timer, mulai/hentikan sesi, ganti mode, lihat analitik, dan terapkan rekomendasi AI.

**Alur data:** Wokwi (ESP32) → EMQX Serverless (MQTT) → web Streamlit yang berjalan di Colab → riwayat sesi di Google Drive.

**Urutan menjalankan (cukup sekali setiap membuka Colab):**
1. Sel 1: install library dan tunnel.
2. Sel 2: isi alamat EMQX dan username; password dari Colab Secrets `MQTT_PASSWORD`; izinkan Google Drive.
3. Sel 3 dan 4: menulis file kode `focuslamp_core.py` dan `app.py` (tinggal dijalankan, tidak perlu diubah).
4. Sel 5: menyalakan web dan menampilkan link. **Buka link-nya sekali**: saat halaman pertama kali dibuka, web mulai tersambung ke EMQX dan mencatat data.
5. Biarkan tab Colab tetap terbuka selama belajar. Tab web boleh ditutup dan dibuka lagi kapan saja.

Notebook lama `FocusLamp_Dashboard.ipynb` tetap bisa dipakai sebagai cadangan; keduanya memakai file riwayat yang sama di Drive.

## 1. Install library dan tunnel
- `streamlit` membuat halaman web dari kode Python.
- `paho-mqtt` menyambung ke EMQX; `pandas`, `plotly`, `scikit-learn` untuk data, grafik, dan AI.
- `cloudflared` membuat **link publik** ke web yang berjalan di dalam Colab (gratis, tanpa akun).

In [ ]:
!pip -q install streamlit==1.66.1 paho-mqtt plotly pandas scikit-learn
![ -f cloudflared ] || (wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared && chmod +x cloudflared)
print("Library dan cloudflared siap.")

## 2. Konfigurasi EMQX dan Google Drive
Isi `BROKER` dengan **Address** dari halaman Overview deployment EMQX dan `USERNAME` dari **Access Control → Authentication**. Password **jangan ditulis di sini**: simpan di Colab Secrets (ikon kunci di sidebar kiri Colab) dengan nama `MQTT_PASSWORD` dan aktifkan akses notebook.

Riwayat sesi disimpan di `My Drive/FocusLamp/riwayat_sesi.csv` supaya data minggu lalu tetap ada untuk rekomendasi AI.

In [ ]:
BROKER    = "ISI_ALAMAT_DEPLOYMENT.emqxsl.com"   # Address deployment EMQX, tanpa mqtts://
PORT      = 8883                                 # MQTT over TLS
USERNAME  = "ISI_USERNAME"
DEVICE_ID = "fl-01"                              # harus sama dengan DEVICE_ID di sketch.ino

SIMPAN_KE_DRIVE = True                           # False = simpan di Colab saja (hilang saat runtime direset)
HISTORY_FILE    = "FocusLamp/riwayat_sesi.csv"   # lokasi di dalam My Drive

import os
try:
    from google.colab import userdata
    PASSWORD = userdata.get("MQTT_PASSWORD")
    print("Password diambil dari Colab Secrets")
except Exception:
    import getpass
    PASSWORD = getpass.getpass("Password MQTT (EMQX): ")

HISTORY_PATH = os.path.abspath(os.path.basename(HISTORY_FILE))
if SIMPAN_KE_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        HISTORY_PATH = os.path.join("/content/drive/MyDrive", HISTORY_FILE)
    except Exception as e:
        print("Google Drive tidak tersedia, riwayat disimpan di Colab:", e)
os.makedirs(os.path.dirname(HISTORY_PATH), exist_ok=True)
print("Riwayat sesi:", HISTORY_PATH)

## 3. Kode inti: `focuslamp_core.py`
Sel ini hanya **menulis file**, tidak menampilkan apa pun. Isinya sama dengan notebook lama, dikelompokkan menjadi lima bagian:
1. **Konstanta**: warna state, label alasan sesi berakhir, pilihan durasi.
2. **`FocusLampHub`**: koneksi MQTT ke EMQX, menerima `status`/`telemetry`/`event`/`session`, menyimpan riwayat ke Drive, mengirim perintah ke `cmd`, dan mode simulasi.
3. **Grafik monitoring**: jarak + ambang pergi, cahaya + kecerahan, riwayat sesi, tabel event.
4. **Analitik** seluruh sesi.
5. **Rekomendasi AI mingguan**: perbandingan mingguan, model peluang bertahan fokus (Kaplan–Meier berbobot waktu), dan Logistic Regression untuk jam belajar terbaik.

In [ ]:
%%writefile focuslamp_core.py
"""FocusLamp core: koneksi MQTT, riwayat sesi, grafik, analitik, dan rekomendasi AI.

Dipakai oleh app.py (tampilan Streamlit). File ini tidak memakai Streamlit sama sekali,
jadi logikanya sama persis dengan notebook Colab dan bisa diuji terpisah.
"""
import json
import math
import os
import random
import ssl
import threading
import time
from datetime import datetime, timedelta, timezone

import numpy as np
import pandas as pd
import paho.mqtt.client as mqtt
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# =====================================================================
# 1. Konstanta
# =====================================================================
WIB = timezone(timedelta(hours=7))
MAX_ROWS = 3000
DURATION_PRESETS = [1, 2, 3, 5, 10, 15, 20, 25, 30, 45, 60]

STATE_COLOR = {"IDLE": "#bdbdbd", "READY": "#90caf9", "FOCUS": "#66bb6a", "AWAY": "#ffca28",
               "SELESAI": "#2e7d32", "END_PAKSA": "#e53935", "DITINGGAL": "#8d6e63"}
STATE_INFO = {"IDLE": "HP belum di dock", "READY": "HP di dock, siap mulai", "FOCUS": "Sesi fokus berjalan",
              "AWAY": "Pengguna meninggalkan meja", "SELESAI": "Sesi selesai", "END_PAKSA": "Sesi dihentikan",
              "DITINGGAL": "Sesi ditinggal"}
RESULT_COLOR = {"SELESAI": "#2e7d32", "END_PAKSA": "#e53935", "DITINGGAL": "#fb8c00"}
REASON_LABEL = {"waktu_habis": "Waktu habis", "tombol_end": "Tombol END", "hp_diangkat": "HP diangkat",
                "ditinggal": "Ditinggal", "dashboard": "Stop dashboard"}
REASON_TIP = {
    "hp_diangkat": "HP paling sering membuat sesi gagal. Letakkan HP di dock sebelum mulai dan aktifkan mode Jangan Ganggu agar tidak tergoda membuka notifikasi.",
    "ditinggal": "Kamu sering meninggalkan meja terlalu lama. Siapkan minum, buku, dan alat tulis sebelum menekan START.",
    "tombol_end": "Kamu sering menghentikan sesi sendiri. Durasinya mungkin terasa terlalu panjang; naikkan durasi bertahap setelah konsisten.",
    "dashboard": "Beberapa sesi dihentikan dari dashboard. Hentikan dari luar hanya bila memang perlu.",
}
SESSION_COLS = ["device", "session_id", "start_ts", "end_ts", "target_s", "focus_s", "completion_ratio", "result",
                "end_reason", "away_count", "away_total_s", "first_away_min", "dock_lift_count", "avg_light_raw",
                "baseline_cm", "demo"]


# =====================================================================
# 2. Hub: koneksi MQTT + penyimpanan data
# =====================================================================
class FocusLampHub:
    """Satu objek untuk semua data FocusLamp: menerima pesan MQTT, menyimpan riwayat, dan mengirim perintah."""

    def __init__(self, broker, port=8883, username="", password="", device_id="fl-01",
                 history_path="riwayat_sesi.csv", use_tls=None, connect=True):
        """Simpan konfigurasi, siapkan penyimpanan data, muat riwayat dari file, lalu sambungkan ke broker MQTT."""
        self.broker, self.port = broker, int(port)
        self.username, self.password = username, password
        self.device_id = device_id
        self.topic_base = f"sic8/focuslamp/{device_id}"
        self.use_tls = (self.port != 1883) if use_tls is None else use_tls
        self.history_path = history_path
        self.lock = threading.Lock()
        self.store = {"status": None, "telemetry": [], "event": [], "session": []}
        self.conn = {"connected": False, "last_error": None, "since": None}
        self.persist = True            # False saat mode simulasi agar data contoh tidak masuk riwayat
        self.mode_simulasi = None
        self.client = None
        os.makedirs(os.path.dirname(history_path) or ".", exist_ok=True)
        self.load_history()
        if connect:
            self.connect()

    # ---------- koneksi ----------
    def configured(self):
        """True bila BROKER dan USERNAME sudah diisi (bukan contoh 'ISI_...')."""
        return bool(self.broker) and not self.broker.startswith("ISI_") and bool(self.username)

    def connect(self):
        """Buat klien MQTT (TLS untuk port 8883) dan mulai menyambung di background; menyambung ulang otomatis bila putus."""
        if not self.configured():
            self.conn["last_error"] = "BROKER / USERNAME belum diisi"
            return
        c = mqtt.Client(mqtt.CallbackAPIVersion.VERSION2,
                        client_id=f"streamlit-{self.device_id}-{int(time.time())}")
        c.username_pw_set(self.username, self.password)
        if self.use_tls:
            c.tls_set(tls_version=ssl.PROTOCOL_TLS_CLIENT)
        c.on_connect, c.on_disconnect, c.on_message = self._on_connect, self._on_disconnect, self._on_message
        c.reconnect_delay_set(min_delay=1, max_delay=30)
        c.connect_async(self.broker, self.port, keepalive=60)   # tidak memblokir; loop menyambung ulang otomatis
        c.loop_start()
        self.client = c

    def disconnect(self):
        """Hentikan loop MQTT dan putuskan koneksi."""
        if self.client:
            self.client.loop_stop()
            self.client.disconnect()
            self.conn["connected"] = False

    def _on_connect(self, client, userdata, flags, reason_code, properties):
        """Dipanggil paho saat koneksi berhasil/gagal; bila berhasil, subscribe ke semua topic perangkat (sic8/focuslamp/<id>/#)."""
        self.conn["connected"] = not reason_code.is_failure
        if reason_code.is_failure:
            self.conn["last_error"] = f"Gagal tersambung: {reason_code} (cek BROKER, USERNAME, PASSWORD)"
        else:
            self.conn["last_error"] = None
            self.conn["since"] = datetime.now(WIB)
            client.subscribe(f"{self.topic_base}/#", qos=1)

    def _on_disconnect(self, client, userdata, flags, reason_code, properties):
        """Dipanggil paho saat koneksi terputus; tandai status MQTT terputus."""
        self.conn["connected"] = False
        if reason_code.is_failure:
            self.conn["last_error"] = f"Terputus: {reason_code}"

    def _on_message(self, client, userdata, msg):
        """Dipanggil paho untuk setiap pesan masuk; teruskan ke handle_message()."""
        self.handle_message(msg.topic, msg.payload, live=True)

    # ---------- data masuk ----------
    def handle_message(self, topic, payload, recv_time=None, live=False):
        """Simpan satu pesan MQTT ke memori sesuai jenis topic (status/telemetry/event/session). Rekap sesi juga ditulis ke file riwayat; bila jam ESP32 belum sinkron, waktu mulai/selesai diisi dari waktu terima. Saat mode data contoh, pesan langsung dari alat tidak ditampilkan, tetapi rekap sesinya tetap disimpan."""
        kind = topic.rsplit("/", 1)[-1]
        try:
            data = json.loads(payload)
        except Exception:
            self.conn["last_error"] = f"payload bukan JSON di {topic}"
            return
        data["_recv"] = recv_time or datetime.now(WIB)
        if kind == "session":
            if not data.get("start_ts") or data["start_ts"] < 1_000_000_000:   # jam ESP32 belum tersinkron NTP
                end = int(data["_recv"].timestamp())
                data["end_ts"] = end
                data["start_ts"] = end - int(data.get("focus_s") or 0) - int(data.get("away_total_s") or 0)
            if self.persist or live:      # sesi asli tetap disimpan walaupun sedang melihat data contoh
                try:
                    self.save_session(data)
                except Exception as e:
                    self.conn["last_error"] = f"gagal menyimpan riwayat: {e}"
        if live and self.mode_simulasi:   # saat mode contoh, data asli tidak dicampur ke tampilan
            return
        with self.lock:
            if kind == "status":
                self.store["status"] = data
            elif kind in ("telemetry", "event", "session"):
                self.store[kind].append(data)
                if len(self.store[kind]) > MAX_ROWS:
                    self.store[kind] = self.store[kind][-MAX_ROWS:]

    # ---------- riwayat permanen ----------
    def save_session(self, data):
        """Tambahkan satu rekap sesi sebagai satu baris baru di file riwayat CSV."""
        row = {c: data.get(c) for c in SESSION_COLS}
        new_file = not os.path.exists(self.history_path)
        pd.DataFrame([row], columns=SESSION_COLS).to_csv(self.history_path, mode="a", header=new_file, index=False)

    def load_history(self):
        """Baca file riwayat CSV ke memori (tanpa duplikat session_id); kembalikan jumlah sesi yang dimuat."""
        if not os.path.exists(self.history_path):
            return 0
        h = pd.read_csv(self.history_path).drop_duplicates("session_id", keep="last")
        rows = []
        for r in h.to_dict("records"):
            r = {k: (None if isinstance(v, float) and math.isnan(v) else v) for k, v in r.items()}
            r["demo"] = str(r.get("demo")).lower() in ("true", "1")
            r["_recv"] = datetime.fromtimestamp(int(r["end_ts"]), WIB) if r.get("end_ts") else datetime.now(WIB)
            rows.append(r)
        with self.lock:
            known = {r["session_id"] for r in rows}
            self.store["session"] = rows + [s for s in self.store["session"] if s.get("session_id") not in known]
        return len(rows)

    def history_count(self):
        """Jumlah sesi unik yang sudah tersimpan di file riwayat."""
        if not os.path.exists(self.history_path):
            return 0
        try:
            return int(pd.read_csv(self.history_path)["session_id"].nunique())
        except Exception:
            return 0

    # ---------- membaca data ----------
    def frames(self):
        """Salin data di memori menjadi DataFrame telemetry, event, session, dan status terakhir (aman dipanggil saat pesan baru masuk)."""
        with self.lock:
            tel = pd.DataFrame(self.store["telemetry"])
            ev = pd.DataFrame(self.store["event"])
            ses = pd.DataFrame(self.store["session"])
            st = dict(self.store["status"]) if self.store["status"] else None
        return tel, ev, ses, st

    def sessions_df(self):
        """Dataset satu baris per sesi, ditambah kolom waktu mulai (WIB), jam mulai, dan nama hari."""
        _, _, ses, _ = self.frames()
        if ses.empty:
            return ses
        ses = ses.drop(columns=["_recv"], errors="ignore").drop_duplicates("session_id", keep="last")
        start = pd.to_datetime(ses["start_ts"].where(ses["start_ts"] > 0), unit="s", utc=True).dt.tz_convert("Asia/Jakarta")
        ses["start_time_wib"] = start.dt.strftime("%Y-%m-%d %H:%M:%S")
        ses["start_hour"] = start.dt.hour
        ses["day_of_week"] = start.dt.day_name()
        return ses.reset_index(drop=True)

    def status_now(self):
        """Ringkasan kondisi alat saat ini untuk kartu status: state, sisa waktu, durasi, hadir, dock, cahaya, mode, waktu update."""
        tel, _, _, st = self.frames()
        last = tel.iloc[-1].to_dict() if not tel.empty else {}
        state = (st or {}).get("state") or last.get("state") or "-"
        status_lebih_baru = st is not None and (not last or st["_recv"] >= last["_recv"])
        remaining = (st.get("remaining_s") if status_lebih_baru else last.get("remaining_s"))
        demo = (st or {}).get("demo", last.get("demo"))
        return {
            "state": state,
            "remaining_s": remaining,
            "target_s": (st or {}).get("target_s"),
            "present": last.get("present"),
            "dock": last.get("dock"),
            "light_pct": last.get("light_pct"),
            "demo": demo,
            "paused": last.get("paused"),
            "updated": max([t for t in (last.get("_recv"), (st or {}).get("_recv")) if t is not None], default=None),
        }

    def countdown(self):
        """Data hitung mundur untuk browser: sisa detik terakhir dari alat, kapan diterima (epoch detik),
        kecepatan timer alat dibanding jam dinding, dan apakah timer sedang berjalan.

        Kecepatan diukur dari telemetry terakhir: di Wokwi simulasi sering berjalan lebih lambat dari waktu
        nyata (misalnya 70%), sehingga 1 detik di alat bisa memakan lebih dari 1 detik jam dinding."""
        tel, _, _, st = self.frames()
        s = self.status_now()
        rem, at = s["remaining_s"], time.time()
        rate = 1.0
        if not tel.empty:
            last = tel.iloc[-1]
            rem, at = last.get("remaining_s"), pd.Timestamp(last["_recv"]).timestamp()
            if st is not None and pd.Timestamp(st["_recv"]).timestamp() > at:
                rem, at = st.get("remaining_s"), pd.Timestamp(st["_recv"]).timestamp()
            t = tel.tail(15)
            t = t[t["state"].isin(["FOCUS", "AWAY"])]
            if len(t) >= 2:
                waktu = t["_recv"].map(lambda x: pd.Timestamp(x).timestamp()).to_numpy()
                sisa = pd.to_numeric(t["remaining_s"], errors="coerce").to_numpy(dtype=float)
                d_sisa, d_waktu = -np.diff(sisa), np.diff(waktu)
                ok = (d_sisa > 0) & (d_waktu > 0.3) & (d_sisa < 30)      # hanya pasangan saat timer berjalan
                if ok.sum() >= 1 and d_waktu[ok].sum() > 1.5:
                    rate = float(np.clip(d_sisa[ok].sum() / d_waktu[ok].sum(), 0.2, 1.5))
        segar = (time.time() - at) < 15
        berjalan = (s["state"] in ("FOCUS", "AWAY") and not s["paused"] and s["dock"] is not False
                    and segar and rem is not None and rem > 0)
        return {"rem": float(rem) if rem is not None else None, "at": at, "rate": round(rate, 2),
                "running": bool(berjalan), "target": s["target_s"], "state": s["state"]}

    # ---------- perintah ke ESP32 ----------
    def send_cmd(self, payload):
        """Kirim satu perintah JSON ke topic cmd; ditolak bila MQTT terputus atau mode simulasi aktif. Mengembalikan (berhasil, pesan)."""
        if self.mode_simulasi:
            return False, "Mode simulasi aktif: perintah tidak dikirim. Klik 'Kembali ke data asli' dulu."
        if not self.client or not self.conn["connected"]:
            return False, "MQTT belum tersambung, perintah tidak terkirim."
        info = self.client.publish(f"{self.topic_base}/cmd", json.dumps(payload), qos=1)
        if info.rc == 0:
            return True, f"Terkirim: {json.dumps(payload)}"
        return False, f"Gagal kirim (rc={info.rc})"

    def set_duration(self, minutes):
        """Perintah ke ESP32: atur durasi sesi berikutnya (menit)."""
        return self.send_cmd({"action": "set_duration", "duration_min": minutes})

    def start_session(self, minutes=None):
        """Perintah ke ESP32: mulai sesi; durasi opsional (menit). Hanya berhasil bila alat READY."""
        return self.send_cmd({"action": "start", **({"duration_min": minutes} if minutes else {})})

    def stop_session(self):
        """Perintah ke ESP32: hentikan sesi (tercatat END_PAKSA dengan alasan dashboard)."""
        return self.send_cmd({"action": "stop"})

    def set_demo(self, on=True):
        """Perintah ke ESP32: ganti mode waktu demo (True) atau normal (False)."""
        return self.send_cmd({"action": "set_mode", "demo": bool(on)})

    # ---------- mode simulasi ----------
    def clear_data(self):
        """Kosongkan data di memori (file riwayat tidak terhapus)."""
        with self.lock:
            self.store.update({"status": None, "telemetry": [], "event": [], "session": []})

    def back_to_real(self):
        """Keluar dari mode simulasi: kosongkan data contoh lalu muat ulang riwayat asli dari file."""
        self.mode_simulasi = None
        self.persist = True
        self.clear_data()
        return self.load_history()

    def simulate_demo(self):
        """Tiga sesi contoh mode demo (SELESAI, END_PAKSA, DITINGGAL) untuk mencoba tampilan."""
        self.clear_data()
        self.persist, self.mode_simulasi = False, "demo"
        base = self.topic_base
        t = datetime.now(WIB) - timedelta(minutes=12)
        for k, (result, aways) in enumerate([("SELESAI", 0), ("END_PAKSA", 1), ("DITINGGAL", 1)]):
            sid = f"{self.device_id}-sim{k + 1}"
            self.handle_message(f"{base}/event", json.dumps({"type": "SESSION_START", "session_id": sid,
                                                              "distance_cm": 50, "elapsed_s": 0}), t)
            focus, light = 0, random.randint(30, 80)
            for step in range(60):
                state, dist = "FOCUS", 48 + random.random() * 6
                if result != "SELESAI" and 20 <= step < (60 if result == "DITINGGAL" else 26):
                    state, dist = "AWAY", 150 + random.random() * 10
                if state == "FOCUS":
                    focus += 2
                self.handle_message(f"{base}/telemetry", json.dumps({
                    "state": state, "distance_cm": round(dist, 1), "threshold_cm": 90, "present": state == "FOCUS",
                    "dock": True, "light_raw": 2000, "light_pct": light, "brightness": int(60 + (100 - light) * 1.4),
                    "elapsed_s": focus, "remaining_s": max(0, 120 - focus), "paused": False, "demo": True}), t)
                if step == 20 and aways:
                    self.handle_message(f"{base}/event", json.dumps({"type": "AWAY_START", "session_id": sid,
                                                                      "distance_cm": round(dist, 1), "elapsed_s": focus}), t)
                t += timedelta(seconds=2)
                if result == "END_PAKSA" and step == 40:
                    break
            self.handle_message(f"{base}/event", json.dumps({"type": "SESSION_END", "session_id": sid,
                                                              "distance_cm": 50, "elapsed_s": focus}), t)
            self.handle_message(f"{base}/session", json.dumps({
                "device": self.device_id, "session_id": sid, "start_ts": int(t.timestamp()) - 120,
                "end_ts": int(t.timestamp()), "target_s": 120, "focus_s": min(focus, 120),
                "completion_ratio": round(min(focus, 120) / 120, 3), "result": result,
                "end_reason": {"SELESAI": "waktu_habis", "END_PAKSA": "tombol_end", "DITINGGAL": "ditinggal"}[result],
                "away_count": aways, "away_total_s": 20 if aways else 0, "first_away_min": 0.67 if aways else None,
                "dock_lift_count": 0, "avg_light_raw": 2000, "baseline_cm": 50.0, "demo": True}), t)
            t += timedelta(seconds=10)
        self.handle_message(f"{base}/status", json.dumps({"state": "READY", "remaining_s": 120, "target_s": 120,
                                                          "demo": True}), t)

    def simulate_history(self, seed=7):
        """Data contoh 2 minggu: minggu 1 timer 20 menit selalu selesai, minggu 2 sering gagal."""
        rng = random.Random(seed)
        self.clear_data()
        self.persist, self.mode_simulasi = False, "2 minggu"
        hari0 = (datetime.now(WIB) - timedelta(days=14)).replace(hour=0, minute=0, second=0, microsecond=0)
        minggu2 = ["SELESAI", "hp_diangkat", "ditinggal", "hp_diangkat", "SELESAI", "tombol_end", "hp_diangkat"]
        for hari in range(14):
            target = 20 * 60
            if hari < 7:
                hasil, jam, alasan = "SELESAI", rng.choice([19, 19, 20]), "waktu_habis"
            else:
                alasan = minggu2[hari - 7]
                hasil = {"SELESAI": "SELESAI", "ditinggal": "DITINGGAL"}.get(alasan, "END_PAKSA")
                alasan = "waktu_habis" if alasan == "SELESAI" else alasan
                jam = rng.choice([19, 20]) if hasil == "SELESAI" else rng.choice([21, 21, 22])
            start = hari0 + timedelta(days=hari, hours=jam, minutes=rng.randint(0, 40))
            fokus = target if hasil == "SELESAI" else rng.randint(11 * 60, 16 * 60)
            pergi = rng.randint(0, 1) if hari < 7 else rng.randint(1, 3)
            away_s = pergi * rng.randint(20, 60)
            end = start + timedelta(seconds=fokus + away_s)
            self.handle_message(f"{self.topic_base}/session", json.dumps({
                "device": self.device_id, "session_id": f"{self.device_id}-hist{hari + 1:02d}",
                "start_ts": int(start.timestamp()), "end_ts": int(end.timestamp()), "target_s": target,
                "focus_s": fokus, "completion_ratio": round(fokus / target, 3), "result": hasil, "end_reason": alasan,
                "away_count": pergi, "away_total_s": away_s,
                "first_away_min": round(rng.uniform(9, 14), 1) if pergi else None,
                "dock_lift_count": int(alasan == "hp_diangkat") + (rng.randint(0, 1) if hari >= 7 else 0),
                "avg_light_raw": rng.randint(1500, 2600), "baseline_cm": 50.0, "demo": False}), end)


# =====================================================================
# 3. Grafik monitoring
# =====================================================================
def fmt_mmss(sec):
    """Ubah detik menjadi teks mm:ss; '-' bila nilai kosong."""
    if sec is None or (isinstance(sec, float) and math.isnan(sec)):
        return "-"
    sec = int(sec)
    return f"{sec // 60:02d}:{sec % 60:02d}"


def _empty_fig(text, height=300):
    """Grafik kosong berisi pesan, dipakai saat data belum ada."""
    fig = go.Figure()
    fig.add_annotation(text=text, x=0.5, y=0.5, xref="paper", yref="paper", showarrow=False,
                       font=dict(size=14, color="#757575"))
    fig.update_layout(height=height, template="plotly_white", xaxis=dict(visible=False), yaxis=dict(visible=False),
                      margin=dict(l=20, r=20, t=40, b=20))
    return fig


def fig_distance(tel):
    """Grafik jarak pengguna terhadap waktu, garis ambang pergi, dan warna latar sesuai state."""
    if tel.empty:
        return _empty_fig("Belum ada telemetry. Jalankan simulasi Wokwi.", 340)
    x = tel["_recv"]
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=x, y=tel["distance_cm"], name="Jarak", mode="lines", line=dict(color="#1f4e79", width=2)))
    if "threshold_cm" in tel:
        fig.add_trace(go.Scatter(x=x, y=tel["threshold_cm"], name="Ambang pergi", mode="lines",
                                 line=dict(color="#e53935", width=1.5, dash="dash")))
    seg_start, seg_state = x.iloc[0], tel["state"].iloc[0]
    for i in range(1, len(tel) + 1):
        if i == len(tel) or tel["state"].iloc[i] != seg_state:
            end = x.iloc[i] if i < len(tel) else x.iloc[-1]
            fig.add_vrect(x0=seg_start, x1=end, fillcolor=STATE_COLOR.get(seg_state, "#eeeeee"), opacity=0.18, line_width=0)
            if i < len(tel):
                seg_start, seg_state = x.iloc[i], tel["state"].iloc[i]
    fig.update_layout(title="Jarak pengguna dan ambang pergi (cm)", height=340, template="plotly_white",
                      legend=dict(orientation="h", y=-0.2), margin=dict(l=50, r=20, t=50, b=40))
    fig.update_yaxes(rangemode="tozero")
    return fig


def fig_light(tel):
    """Grafik cahaya ruangan (%) dan kecerahan lampu (%) terhadap waktu."""
    if tel.empty:
        return _empty_fig("Belum ada data cahaya.")
    x = tel["_recv"]
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=x, y=tel["light_pct"], name="Cahaya ruangan", mode="lines", line=dict(color="#f9a825", width=2)))
    fig.add_trace(go.Scatter(x=x, y=tel["brightness"] / 255 * 100, name="Kecerahan lampu", mode="lines",
                             line=dict(color="#6a1b9a", width=2)))
    fig.update_layout(title="Cahaya ruangan & kecerahan lampu (%)", height=300, template="plotly_white",
                      legend=dict(orientation="h", y=-0.25), margin=dict(l=50, r=20, t=50, b=40))
    fig.update_yaxes(range=[0, 105])
    return fig


def fig_history(sdf):
    """Grafik batang waktu fokus tiap sesi (warna sesuai hasil) dengan penanda target durasi."""
    if sdf.empty:
        return _empty_fig("Belum ada sesi yang selesai.")
    labels = [f"#{i + 1}" for i in range(len(sdf))]
    fig = go.Figure()
    for res, color in RESULT_COLOR.items():
        m = sdf["result"] == res
        if m.any():
            fig.add_trace(go.Bar(x=[l for l, k in zip(labels, m) if k], y=sdf.loc[m, "focus_s"], name=res, marker_color=color))
    fig.add_trace(go.Scatter(x=labels, y=sdf["target_s"], name="Target", mode="markers",
                             marker=dict(symbol="line-ew-open", size=22, color="#424242", line=dict(width=2))))
    fig.update_layout(title="Riwayat sesi: waktu fokus vs target (detik)", height=300, template="plotly_white",
                      barmode="overlay", legend=dict(orientation="h", y=-0.25), margin=dict(l=50, r=20, t=50, b=40))
    fig.update_xaxes(categoryorder="array", categoryarray=labels)
    return fig


def events_table(ev, n=10):
    """Tabel n event terbaru (terbaru di atas) untuk tab Monitoring."""
    if ev.empty:
        return pd.DataFrame(columns=["Waktu", "Event", "Jarak (cm)", "Detik fokus"])
    tail = ev.tail(n).iloc[::-1]
    return pd.DataFrame({"Waktu": tail["_recv"].dt.strftime("%H:%M:%S"), "Event": tail["type"],
                         "Jarak (cm)": tail.get("distance_cm", pd.Series(dtype=float)).round(1),
                         "Detik fokus": tail.get("elapsed_s")}).reset_index(drop=True)


# =====================================================================
# 4. Analitik semua sesi
# =====================================================================
def analytics_summary(df):
    """Angka ringkasan semua sesi: jumlah, tingkat selesai, ketercapaian target, total menit fokus, rata-rata pergi."""
    n = len(df)
    return {"sesi": n,
            "tingkat_selesai": (df["result"] == "SELESAI").mean() * 100 if n else 0,
            "ketercapaian": df["completion_ratio"].mean() * 100 if n else 0,
            "fokus_menit": df["focus_s"].sum() / 60 if n else 0,
            "pergi_per_sesi": df["away_count"].mean() if n else 0}


def fig_analytics(df):
    """Empat grafik analitik: distribusi hasil, penyebab sesi berakhir, tren waktu fokus, tingkat selesai per jam mulai."""
    if df.empty:
        return None
    n = len(df)
    fig = make_subplots(rows=2, cols=2, vertical_spacing=0.16, horizontal_spacing=0.16,
                        subplot_titles=("Distribusi hasil sesi", "Penyebab sesi berakhir",
                                        "Tren waktu fokus per sesi (menit)", "Tingkat selesai per jam mulai (%)"))
    counts = df["result"].value_counts()
    fig.add_trace(go.Bar(x=counts.index, y=counts.values, marker_color=[RESULT_COLOR.get(r, "#999") for r in counts.index],
                         text=counts.values, textposition="outside", showlegend=False), row=1, col=1)
    reasons = df["end_reason"].map(REASON_LABEL).fillna(df["end_reason"]).value_counts().sort_values()
    fig.add_trace(go.Bar(x=reasons.values, y=reasons.index, orientation="h", marker_color="#4c78a8",
                         text=reasons.values, textposition="outside", showlegend=False), row=1, col=2)
    labels = [f"#{i + 1}" for i in range(n)]
    fig.add_trace(go.Scatter(x=labels, y=df["focus_s"] / 60, mode="lines+markers", name="Waktu fokus",
                             line=dict(color="#2e7d32", width=2)), row=2, col=1)
    fig.add_trace(go.Scatter(x=labels, y=df["target_s"] / 60, mode="lines", name="Target",
                             line=dict(color="#757575", width=1.5, dash="dash")), row=2, col=1)
    if df["start_hour"].notna().any():
        by_hour = df.dropna(subset=["start_hour"]).groupby("start_hour")["result"].apply(lambda r: (r == "SELESAI").mean() * 100)
        fig.add_trace(go.Bar(x=[f"{int(h):02d}:00" for h in by_hour.index], y=by_hour.values, marker_color="#f9a825",
                             showlegend=False, text=[f"{v:.0f}%" for v in by_hour.values], textposition="outside"), row=2, col=2)
        fig.update_yaxes(range=[0, 115], row=2, col=2)
    fig.update_yaxes(range=[0, counts.max() * 1.25], row=1, col=1)
    fig.update_xaxes(range=[0, reasons.max() * 1.25], row=1, col=2)
    fig.update_yaxes(rangemode="tozero", row=2, col=1)
    fig.update_layout(height=760, template="plotly_white", legend=dict(orientation="h", y=-0.08),
                      margin=dict(l=60, r=40, t=50, b=40))
    return fig


# =====================================================================
# 5. Rekomendasi AI mingguan
# =====================================================================
AMBANG_BERTAHAN = 0.60     # durasi aman = peluang masih fokus >= 60%
PARUH_HARI = 7             # sesi 7 hari lalu berbobot setengah dari sesi hari ini
DURASI_MAX = 60


def _is_demo(s):
    """True untuk baris yang berasal dari mode demo (kolom demo bisa bool atau teks)."""
    return s.astype(str).str.lower().isin(["true", "1"])


def _bulat(m, step):
    """Bulatkan menit ke kelipatan step terdekat (minimal satu step)."""
    return int(max(step, step * round(m / step)))


def _fmt_jam(j):
    """Ubah jam desimal (misal 19.5) menjadi teks HH:MM."""
    return f"{int(j):02d}:{int(round((j % 1) * 60)) % 60:02d}"


def _prep(df):
    """Tambahkan kolom analisis: minggu ke-, durasi & fokus (menit), selesai/gagal, jam mulai, cahaya, dan bobot (sesi lama berbobot lebih kecil)."""
    d = df.copy()
    d["start"] = pd.to_datetime(d["start_ts"], unit="s", utc=True).dt.tz_convert("Asia/Jakarta")
    hari0 = d["start"].dt.normalize().min()
    d["minggu"] = ((d["start"].dt.normalize() - hari0).dt.days // 7 + 1).astype(int)
    d["target_min"] = d["target_s"] / 60
    d["fokus_min"] = d["focus_s"] / 60
    d["selesai"] = d["result"] == "SELESAI"
    d["gagal"] = (~d["selesai"]) & (d["end_reason"] != "dashboard")   # stop dari dashboard bukan kegagalan fokus
    d["jam"] = d["start"].dt.hour + d["start"].dt.minute / 60
    d["cahaya"] = pd.to_numeric(d["avg_light_raw"], errors="coerce")
    d["cahaya"] = d["cahaya"].fillna(d["cahaya"].median() if d["cahaya"].notna().any() else 2000)
    umur_hari = (d["start"].max() - d["start"]).dt.total_seconds() / 86400
    d["bobot"] = 0.5 ** (umur_hari / PARUH_HARI)
    return d.sort_values("start").reset_index(drop=True)


def ringkasan_mingguan(d):
    """Tabel ringkasan per minggu: jumlah sesi, durasi, tingkat selesai, ketercapaian, rata-rata fokus, pergi, HP diangkat, jam mulai."""
    return d.groupby("minggu").agg(sesi=("session_id", "count"), durasi_menit=("target_min", "median"),
                                   tingkat_selesai=("selesai", "mean"), ketercapaian=("completion_ratio", "mean"),
                                   fokus_menit=("fokus_min", "mean"), pergi_per_sesi=("away_count", "mean"),
                                   hp_diangkat=("dock_lift_count", "sum"), jam_mulai=("jam", "mean"))


def kurva_bertahan(d):
    """Kaplan-Meier berbobot: peluang masih fokus sampai menit ke-t.
    Sesi SELESAI dianggap tersensor di durasinya (tidak diketahui apakah sanggup lebih lama)."""
    t = np.where(d["gagal"], d["fokus_min"], d["target_min"]).astype(float)
    e = d["gagal"].to_numpy()
    w = d["bobot"].to_numpy()
    grid = np.round(np.arange(0, DURASI_MAX + 0.25, 0.25), 2)
    S = np.ones_like(grid)
    s = 1.0
    for ti in np.unique(t[e]):
        n_risk = w[t >= ti].sum()
        n_fail = w[(t == ti) & e].sum()
        if n_risk > 0:
            s *= 1 - n_fail / n_risk
        S[grid >= ti] = s
    return grid, S, float(t.max())


def peluang_di(grid, S, menit):
    """Ambil nilai kurva peluang bertahan fokus pada menit tertentu."""
    return float(S[min(np.searchsorted(grid, menit, side="right") - 1, len(S) - 1)])


def model_jam(d, durasi):
    """Logistic Regression: peluang sesi selesai berdasarkan jam mulai, durasi, dan cahaya."""
    y = d["selesai"].astype(int)
    if len(d) < 12 or y.nunique() < 2 or d["start"].dt.hour.nunique() < 2:
        return None

    def fitur(jam, durasi_min, cahaya):
        """Susun fitur model: jam mulai (sin/cos agar 23:00 dekat dengan 00:00), durasi, dan cahaya."""
        jam = np.asarray(jam, dtype=float)
        return pd.DataFrame({"jam_sin": np.sin(2 * np.pi * jam / 24), "jam_cos": np.cos(2 * np.pi * jam / 24),
                             "durasi": np.broadcast_to(durasi_min, jam.shape).astype(float),
                             "cahaya": np.broadcast_to(cahaya, jam.shape).astype(float)})

    X = fitur(d["jam"], d["target_min"].to_numpy(), d["cahaya"].to_numpy())
    model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
    k = int(min(5, y.value_counts().min()))
    akurasi = (cross_val_score(model, X, y, cv=StratifiedKFold(k, shuffle=True, random_state=0)).mean()
               if k >= 2 else float("nan"))
    model.fit(X, y, logisticregression__sample_weight=d["bobot"].to_numpy())
    jam_kandidat = np.array(sorted(d["start"].dt.hour.unique()), dtype=float)
    peluang = model.predict_proba(fitur(jam_kandidat, durasi, d["cahaya"].median()))[:, 1]
    return {"jam": jam_kandidat.astype(int), "peluang": peluang, "akurasi": akurasi, "n": len(d)}


def hitung_rekomendasi(df, pakai_demo=False):
    """Hasilkan rekomendasi durasi timer + alasan. Mengembalikan dict; kunci 'ok' False bila data belum cukup."""
    if df.empty:
        return {"ok": False, "pesan": "Belum ada sesi. Jalankan beberapa sesi di Wokwi, atau coba simulasi 2 minggu di sidebar."}
    if not pakai_demo:
        if _is_demo(df["demo"]).all():
            return {"ok": False, "pesan": "Semua sesi masih mode demo. Pakai mode normal di tab Kontrol untuk data asli, "
                                          "atau nyalakan 'Ikutkan sesi mode demo' untuk uji coba."}
        df = df[~_is_demo(df["demo"])]
    if len(df) < 3:
        return {"ok": False, "pesan": f"Baru {len(df)} sesi; butuh minimal 3 sesi."}

    d = _prep(df)
    mingguan = ringkasan_mingguan(d)
    step = 5 if d["target_min"].median() >= 10 else 1           # data demo memakai langkah 1 menit
    W = int(d["minggu"].max())
    akhir = d[d["minggu"] == W]
    if len(akhir) < 3:                                          # minggu baru berjalan: pakai 7 sesi terakhir
        akhir = d.tail(7)
    T = _bulat(akhir["target_min"].mode().iloc[0], step)
    di_T = akhir[(akhir["target_min"] - T).abs() < step / 2]
    rate = di_T["selesai"].mean()
    grid, S, horizon = kurva_bertahan(d)
    durasi_aman = max([m for m in range(step, DURASI_MAX + 1, step)
                       if m <= horizon + 1e-9 and peluang_di(grid, S, m) >= AMBANG_BERTAHAN] or [step])

    poin = []
    if rate >= 0.8:
        durasi = min(T + step, DURASI_MAX)
        poin.append(f"**{rate:.0%}** sesi {T} menit di minggu ke-{W} selesai penuh. Kamu sudah konsisten, coba naikkan ke **{durasi} menit**.")
    elif rate >= 0.5:
        durasi = T
        poin.append(f"Baru **{rate:.0%}** sesi {T} menit yang selesai di minggu ke-{W}. Pertahankan **{T} menit** sampai minimal 80% sesi selesai.")
    else:
        berhenti = di_T.loc[di_T["gagal"], "fokus_min"].mean()
        durasi = max(step, min(durasi_aman, T - step))
        teks_berhenti = f" Rata-rata kamu berhenti di menit ke-**{berhenti:.0f}**." if pd.notna(berhenti) else ""
        poin.append(f"Hanya **{rate:.0%}** sesi {T} menit yang selesai di minggu ke-{W}.{teks_berhenti} "
                    f"Model memperkirakan peluang kamu bertahan fokus sampai menit ke-{durasi} sebesar "
                    f"**{peluang_di(grid, S, durasi):.0%}**, jadi turunkan dulu ke **{durasi} menit**, lalu naikkan lagi saat sudah konsisten.")

    lalu = d[(d["minggu"] == W - 1) & ((d["target_min"] - T).abs() < step / 2)]
    if len(lalu) >= 3 and W > 1:
        rate_lalu = lalu["selesai"].mean()
        arah = "turun" if rate < rate_lalu - 0.1 else "naik" if rate > rate_lalu + 0.1 else "stabil"
        teks = f"Dibanding minggu ke-{W - 1}, tingkat selesai durasi {T} menit **{arah}** dari {rate_lalu:.0%} menjadi {rate:.0%}."
        if arah == "turun":
            sebab = []
            jl, jk = lalu["jam"].mean(), di_T["jam"].mean()
            if jk - jl >= 1:
                sebab.append(f"rata-rata mulai belajar lebih malam ({_fmt_jam(jl)} → {_fmt_jam(jk)})")
            hl, hk = lalu["dock_lift_count"].mean(), di_T["dock_lift_count"].mean()
            if hk - hl >= 0.3:
                sebab.append(f"HP lebih sering diangkat ({hl:.1f} → {hk:.1f} kali per sesi)")
            pl, pk = lalu["away_count"].mean(), di_T["away_count"].mean()
            if pk - pl >= 0.5:
                sebab.append(f"lebih sering meninggalkan meja ({pl:.1f} → {pk:.1f} kali per sesi)")
            if sebab:
                teks += " Perubahan yang terlihat: " + "; ".join(sebab) + "."
        poin.append(teks)

    mj = model_jam(d, durasi)
    if mj is not None:
        i = int(np.argmax(mj["peluang"]))
        poin.append(f"Jam mulai terbaik menurut model: sekitar **{mj['jam'][i]:02d}:00** (peluang selesai "
                    f"{mj['peluang'][i]:.0%} untuk {durasi} menit; akurasi validasi silang {mj['akurasi']:.0%} dari {mj['n']} sesi).")
    else:
        per_jam = d.groupby(d["start"].dt.hour)["selesai"].agg(["mean", "count"])
        per_jam = per_jam[per_jam["count"] >= 2]
        if len(per_jam) >= 2:
            h = int(per_jam["mean"].idxmax())
            poin.append(f"Tingkat selesai tertinggi saat mulai jam **{h:02d}:00** ({per_jam.loc[h, 'mean']:.0%}). "
                        "Model klasifikasi aktif setelah ada ≥ 12 sesi.")

    gagal_akhir = akhir[akhir["gagal"]]
    if len(gagal_akhir) >= 2:
        utama = gagal_akhir["end_reason"].value_counts()
        if utama.index[0] in REASON_TIP:
            poin.append(f"Penyebab utama sesi gagal minggu ini: **{REASON_LABEL.get(utama.index[0], utama.index[0])}** "
                        f"({int(utama.iloc[0])} dari {len(gagal_akhir)} sesi). {REASON_TIP[utama.index[0]]}")
    pertama_pergi = pd.to_numeric(akhir["first_away_min"], errors="coerce").dropna()
    if len(pertama_pergi) >= 2 and pertama_pergi.median() < T:
        m = pertama_pergi.median()
        poin.append(f"Biasanya kamu mulai meninggalkan meja di menit ke-**{m:.0f}**. Siapkan semua kebutuhan sebelum mulai; "
                    f"kalau tetap sulit, pakai pola {_bulat(m, step)} menit fokus lalu jeda 5 menit.")

    tabel = mingguan.copy()
    tabel["tingkat_selesai"] = (tabel["tingkat_selesai"] * 100).round(0).astype(int).astype(str) + "%"
    tabel["ketercapaian"] = (tabel["ketercapaian"] * 100).round(0).astype(int).astype(str) + "%"
    tabel["jam_mulai"] = tabel["jam_mulai"].map(_fmt_jam)
    tabel = tabel.round(1).rename_axis("minggu ke-")

    return {"ok": True, "durasi": durasi, "durasi_sekarang": T, "minggu": W, "tingkat_selesai": rate,
            "durasi_aman": durasi_aman, "poin": poin, "tabel": tabel, "n": len(d),
            "_plot": {"mingguan": mingguan, "grid": grid, "S": S, "horizon": horizon, "mj": mj, "d": d}}


def fig_rekomendasi(res):
    """Empat grafik rekomendasi: tingkat selesai per minggu, fokus vs durasi, kurva peluang bertahan, peluang selesai per jam."""
    p = res["_plot"]
    mingguan, grid, S, horizon, mj, d = p["mingguan"], p["grid"], p["S"], p["horizon"], p["mj"], p["d"]
    durasi = res["durasi"]
    fig = make_subplots(rows=2, cols=2, vertical_spacing=0.18, horizontal_spacing=0.12,
                        subplot_titles=("Tingkat selesai per minggu (%)", "Rata-rata waktu fokus vs durasi (menit)",
                                        "Peluang bertahan fokus sampai menit ke-t",
                                        f"Peluang sesi selesai per jam mulai (timer {durasi} menit)"))
    xw = [f"Minggu {w}" for w in mingguan.index]
    fig.add_trace(go.Bar(x=xw, y=mingguan["tingkat_selesai"] * 100, marker_color=[
        "#2e7d32" if v >= 0.8 else "#f9a825" if v >= 0.5 else "#e53935" for v in mingguan["tingkat_selesai"]],
        text=[f"{v:.0%}" for v in mingguan["tingkat_selesai"]], textposition="outside", showlegend=False), row=1, col=1)
    fig.add_hline(y=80, line_dash="dot", line_color="#2e7d32", row=1, col=1)
    fig.add_trace(go.Bar(x=xw, y=mingguan["fokus_menit"], name="Rata-rata fokus", marker_color="#4c78a8",
                         text=[f"{v:.1f}" for v in mingguan["fokus_menit"]], textposition="outside"), row=1, col=2)
    fig.add_trace(go.Scatter(x=xw, y=mingguan["durasi_menit"], name="Durasi timer", mode="markers",
                             marker=dict(symbol="line-ew-open", size=40, color="#424242", line=dict(width=3))), row=1, col=2)
    batas = min(DURASI_MAX, max(horizon, durasi) + 5)
    m = grid <= batas
    fig.add_trace(go.Scatter(x=grid[m], y=S[m] * 100, mode="lines", line=dict(color="#6a1b9a", width=3, shape="hv"),
                             name="Peluang bertahan"), row=2, col=1)
    fig.add_hline(y=AMBANG_BERTAHAN * 100, line_dash="dot", line_color="#757575", row=2, col=1)
    fig.add_trace(go.Scatter(x=[durasi], y=[peluang_di(grid, S, durasi) * 100], mode="markers+text",
                             marker=dict(size=13, color="#e53935"), text=[f"rekomendasi {durasi} mnt"],
                             textposition="top right", showlegend=False), row=2, col=1)
    if mj is not None:
        fig.add_trace(go.Bar(x=[f"{h:02d}:00" for h in mj["jam"]], y=mj["peluang"] * 100, marker_color="#f9a825",
                             text=[f"{v:.0%}" for v in mj["peluang"]], textposition="outside", showlegend=False), row=2, col=2)
    else:
        per_jam = d.groupby(d["start"].dt.hour)["selesai"].mean()
        fig.add_trace(go.Bar(x=[f"{h:02d}:00" for h in per_jam.index], y=per_jam.values * 100, marker_color="#f9a825",
                             text=[f"{v:.0%}" for v in per_jam.values], textposition="outside", showlegend=False), row=2, col=2)
    fig.update_yaxes(range=[0, 115], row=1, col=1)
    fig.update_yaxes(range=[0, max(mingguan["durasi_menit"].max(), mingguan["fokus_menit"].max()) * 1.3], row=1, col=2)
    fig.update_yaxes(range=[0, 105], row=2, col=1)
    fig.update_xaxes(title_text="menit", row=2, col=1)
    fig.update_yaxes(range=[0, 115], row=2, col=2)
    fig.update_layout(height=720, template="plotly_white", barmode="overlay",
                      legend=dict(orientation="h", y=-0.1), margin=dict(l=60, r=40, t=50, b=50))
    return fig

## 4. Tampilan web: `app.py`
Sel ini juga hanya menulis file. Halaman web terdiri dari:
Tampilan dibuat untuk pengguna awam, dengan bahasa sehari-hari:
- **Beranda**: status besar dengan cincin hitung mundur yang berjalan mulus di browser (warnanya mengikuti lampu), pilihan lama belajar, tombol **Mulai fokus** dan **Hentikan sesi**. Grafik sensor ada di bagian *Detail sensor (untuk teknisi dan juri)*.
- **Riwayat**: ringkasan 7 hari terakhir, menit fokus per hari, tabel sesi, dan unduhan CSV.
- **Saran AI**: durasi belajar yang disarankan untuk minggu berikutnya, alasannya, dan tombol untuk langsung memakainya.
- **Bantuan**: cara pakai, arti warna lampu, cara memasang web di HP, data contoh, dan info teknis.

Tampilannya **responsif**: di laptop berbentuk dashboard web dua kolom, di HP berubah seperti aplikasi (menu pindah ke bawah layar, cincin timer besar di tengah). Sel kedua di bagian ini membuat ikon dan *manifest* sehingga web bisa dipasang di layar utama HP seperti aplikasi.

In [ ]:
%%writefile app.py
"""FocusLamp: web pendamping lampu belajar (Streamlit).

Dibuat untuk pengguna awam: bahasa sehari-hari, satu status besar di atas, dan tombol yang jelas.
Grafik sensor dan info teknis tetap ada, tetapi disimpan di bagian yang bisa dibuka.

Jalankan dari notebook FocusLamp_Streamlit_Colab.ipynb, atau di laptop:
    streamlit run app.py
Konfigurasi dibaca dari environment variable (diisi notebook) atau .streamlit/secrets.toml:
    FL_BROKER, FL_PORT, FL_USERNAME, FL_PASSWORD, FL_DEVICE_ID, FL_HISTORY_PATH
"""
import os
import time
from datetime import datetime, timedelta

import pandas as pd
import plotly.graph_objects as go
import streamlit as st
import streamlit.components.v1 as components

import focuslamp_core as fl

st.set_page_config(page_title="FocusLamp", page_icon=":material/lightbulb:", layout="wide", initial_sidebar_state="collapsed")

# ---------------------------------------------------------------------
# Gaya tampilan: palet "belajar malam" (tinta biru + cahaya lampu kuning)
# ---------------------------------------------------------------------
INK, LAMP, PAPER, MIST, MUTED = "#22304F", "#F2B33D", "#F7F8FB", "#E6EBF4", "#5B6478"
STYLE = """
<link href="https://fonts.googleapis.com/css2?family=Plus+Jakarta+Sans:wght@400;500;600;700;800&display=swap" rel="stylesheet">
<style>
.stApp, .stApp *:not([data-testid="stIconMaterial"]):not(code):not(pre) {
  font-family: 'Plus Jakarta Sans', system-ui, sans-serif;
}
.stApp { background: #F7F8FB; }
header[data-testid="stHeader"] { display: none; }
.block-container { padding-top: 1.4rem; max-width: 1100px; }
.fl-top { display:flex; justify-content:space-between; align-items:center; gap:12px; flex-wrap:wrap; margin-bottom:6px; }
.fl-brand { font-size:1.55rem; font-weight:800; color:#22304F; letter-spacing:-0.01em; }
.fl-brand small { display:block; font-size:0.9rem; font-weight:500; color:#5B6478; letter-spacing:0; }
.fl-pill { padding:6px 14px; border-radius:999px; font-weight:600; font-size:0.9rem; white-space:nowrap; display:inline-flex; align-items:center; gap:8px; }
.fl-pdot { width:9px; height:9px; border-radius:50%; display:inline-block; }
.fl-advice { background:#FFF7E6; border-left:6px solid #F2B33D; border-radius:18px; padding:22px 26px; margin:6px 0 14px 0; }
.fl-advice .big { font-size:1.7rem; font-weight:800; color:#22304F; line-height:1.2; }
.fl-advice .sub { color:#4A5368; margin-top:6px; font-size:1.02rem; }
.fl-legend { display:flex; align-items:center; gap:10px; margin:4px 0; }
.fl-dot { width:16px; height:16px; border-radius:50%; flex:0 0 16px; border:1px solid rgba(0,0,0,0.12); }
/* Kotak angka ringkasan: --n kolom di laptop, --m kolom di HP */
.fl-label { font-weight:700; color:#22304F; margin:14px 0 0 0; }
.fl-stats { display:grid; grid-template-columns:repeat(var(--n), minmax(0, 1fr)); gap:12px; margin:8px 0 14px 0; }
.fl-stat { background:#fff; border:1px solid #E6EBF4; border-radius:16px; padding:14px 16px; min-width:0; }
.fl-stat .k { font-size:0.86rem; color:#5B6478; }
.fl-stat .v { font-size:1.7rem; font-weight:800; color:#22304F; line-height:1.2; margin-top:4px;
              font-variant-numeric:tabular-nums; white-space:nowrap; overflow:hidden; text-overflow:ellipsis; }
.fl-stat .v small { font-size:0.9rem; font-weight:600; color:#5B6478; }
/* Daftar sesi berbentuk kartu (enak dibaca di HP, tanpa geser ke samping) */
.fl-sesi { background:#fff; border:1px solid #E6EBF4; border-radius:16px; padding:12px 16px; margin-bottom:8px; }
.fl-sesi .top { display:flex; justify-content:space-between; align-items:center; gap:10px; }
.fl-sesi .d { font-weight:700; color:#22304F; }
.fl-sesi .r { color:#5B6478; font-size:0.88rem; margin-top:4px; }
.fl-sesi .bar { height:6px; background:#E6EBF4; border-radius:99px; margin-top:8px; overflow:hidden; }
.fl-sesi .bar span { display:block; height:100%; border-radius:99px; }
.fl-badge { padding:3px 10px; border-radius:999px; font-size:0.8rem; font-weight:700; white-space:nowrap; }
/* Kartu putih (st.container(border=True)) untuk mengelompokkan isi, seperti dashboard web */
div[data-testid="stVerticalBlock"][class*="BorderWrapper"], div[data-testid="stVerticalBlockBorderWrapper"] {
  background:#fff; border-radius:20px !important; border-color:#E6EBF4 !important; }
.fl-mini { background:#FFF7E6; border-radius:16px; padding:14px 16px; margin-top:10px; }
.fl-mini .k { font-size:0.86rem; color:#5B6478; }
.fl-mini .v { font-size:1.25rem; font-weight:800; color:#22304F; margin-top:2px; }
.fl-mobile-only { display:none; }
/* ---------- Tampilan HP: seperti aplikasi (menu di bawah layar) ---------- */
@media (max-width: 640px) {
  .fl-mobile-only { display:block; }
  .fl-desktop-only { display:none; }
  .block-container { padding-left:1rem; padding-right:1rem; padding-top:0.9rem; padding-bottom:110px !important; }
  .fl-brand { font-size:1.3rem; }
  .fl-brand small { display:none; }            /* bilah atas ringkas seperti aplikasi */
  .fl-top { flex-wrap:nowrap; margin-bottom:10px; }
  .fl-pill { font-size:0.78rem; padding:5px 11px; }
  /* Menu tab dipindah menjadi navigasi bawah (bottom navigation) seperti aplikasi Android */
  .stTabs [role="tablist"] {
    position:fixed; left:0; right:0; bottom:0; z-index:999990; gap:0; margin:0;
    background:#fff; border-top:1px solid #E6EBF4; box-shadow:0 -6px 22px rgba(34,48,79,0.10);
    padding:6px 6px calc(8px + env(safe-area-inset-bottom)); overflow:visible; }
  .stTabs [role="tab"] { flex:1 1 0; justify-content:center; padding:4px 2px; min-width:0; height:auto; }
  .stTabs [role="tab"] [data-testid="stMarkdownContainer"] p {
    display:flex; flex-direction:column; align-items:center; gap:3px; font-size:0.72rem; font-weight:600;
    white-space:nowrap; color:#5B6478; margin:0; }
  .stTabs [role="tab"] p > span[role="img"] {
    display:block !important; font-size:1.45rem; line-height:1; padding:4px 16px; border-radius:999px; color:#5B6478; }
  .stTabs [aria-selected="true"] [data-testid="stMarkdownContainer"] p { color:#22304F; font-weight:800; }
  .stTabs [aria-selected="true"] p > span[role="img"] { background:#FFE7B0; color:#22304F; }
  .stTabs .react-aria-SelectionIndicator { display:none; }
  [data-testid="stToastContainer"] { bottom:88px !important; }
  .fl-stats { grid-template-columns:repeat(var(--m), minmax(0, 1fr)); gap:8px; }
  .fl-stat { padding:10px 12px; border-radius:14px; }
  .fl-stat .k { font-size:0.74rem; }
  .fl-stat .v { font-size:1.3rem; }
  .fl-advice { padding:16px 18px; }
  .fl-advice .big { font-size:1.3rem; }
  .fl-advice .sub { font-size:0.95rem; }
  /* tombol selebar layar supaya mudah ditekan jari */
  [data-testid="stElementContainer"]:has(.stButton), [data-testid="stElementContainer"]:has(.stDownloadButton) { width:100% !important; }
  .stButton, .stDownloadButton, .stButton button, .stDownloadButton button { width:100% !important; }
  .stButton button, .stDownloadButton button { min-height:48px; border-radius:14px; }
  h3 { font-size:1.25rem !important; }
}
</style>
"""
st.markdown(STYLE, unsafe_allow_html=True)

# Supaya web bisa dipasang di layar utama HP seperti aplikasi (Chrome: menu > Tambahkan ke layar utama / Instal aplikasi).
# File manifest dan ikon ada di folder static/ dan disajikan Streamlit di alamat app/static/ (server.enableStaticServing).
PWA_JS = """<script>
(function () {
  const d = document, h = d.head;
  if (d.getElementById("fl-manifest")) return;
  const add = (tag, attrs) => { const e = d.createElement(tag); Object.entries(attrs).forEach(([k, v]) => e.setAttribute(k, v)); h.appendChild(e); };
  add("link", {id: "fl-manifest", rel: "manifest", href: "app/static/manifest.json"});
  add("link", {rel: "apple-touch-icon", href: "app/static/icon-192.png"});
  add("meta", {name: "theme-color", content: "#22304F"});
  add("meta", {name: "mobile-web-app-capable", content: "yes"});
  add("meta", {name: "apple-mobile-web-app-capable", content: "yes"});
  add("meta", {name: "apple-mobile-web-app-title", content: "FocusLamp"});
  // Seperti aplikasi: pindah menu di navigasi bawah langsung kembali ke bagian atas halaman.
  d.addEventListener("click", (e) => {
    if (!e.target.closest('[role="tab"]')) return;
    [d.scrollingElement, d.querySelector('[data-testid="stMain"]'), d.querySelector('[data-testid="stAppViewContainer"]')]
      .forEach((el) => { if (el) el.scrollTo({top: 0}); });
  }, true);
})();
</script>"""
st.html(PWA_JS, unsafe_allow_javascript=True)


# ---------------------------------------------------------------------
# Konfigurasi + koneksi (dibuat sekali, dipakai semua pengunjung halaman)
# ---------------------------------------------------------------------
def cfg(key, default=""):
    """Ambil konfigurasi dari environment variable (diisi notebook), lalu dari Streamlit Secrets, atau nilai bawaan."""
    if os.environ.get(key):
        return os.environ[key]
    try:
        return str(st.secrets[key])
    except Exception:
        return default


@st.cache_resource
def get_hub():
    """Buat satu FocusLampHub untuk seluruh aplikasi (disimpan cache sehingga koneksi MQTT tidak dibuat ulang setiap klik)."""
    return fl.FocusLampHub(
        broker=cfg("FL_BROKER"), port=int(cfg("FL_PORT", "8883")),
        username=cfg("FL_USERNAME"), password=cfg("FL_PASSWORD"),
        device_id=cfg("FL_DEVICE_ID", "fl-01"), history_path=cfg("FL_HISTORY_PATH", "riwayat_sesi.csv"))


hub = get_hub()
PRESETS = [15, 20, 25, 30, 45, 60]          # mode normal
PRESETS_UJI = [1, 2, 3, 5, 10, 15]          # mode uji coba (demo): durasi pendek untuk pengujian
HASIL = {"SELESAI": "Tuntas", "END_PAKSA": "Dihentikan", "DITINGGAL": "Ditinggal"}
PENYEBAB = {"waktu_habis": "Waktu habis", "tombol_end": "Tombol berhenti ditekan", "hp_diangkat": "HP diangkat terlalu lama",
            "ditinggal": "Meninggalkan meja terlalu lama", "dashboard": "Dihentikan dari web"}
WARNA_HASIL = {"Tuntas": ("#E3F6EC", "#1E8E55"), "Dihentikan": ("#FDE8E8", "#C23A3E"), "Ditinggal": ("#FDF0E1", "#B86414")}
BULAN = ["", "Jan", "Feb", "Mar", "Apr", "Mei", "Jun", "Jul", "Agu", "Sep", "Okt", "Nov", "Des"]
PLOT_CFG = {"displayModeBar": False, "scrollZoom": False, "responsive": True}


def tgl_id(d):
    """Format tanggal singkat berbahasa Indonesia, misalnya 24 Okt."""
    return f"{d.day} {BULAN[d.month]}"


def angka(items, n, m=None):
    """Baris kotak angka ringkasan. items = [(label, nilai)], n = kolom di laptop, m = kolom di HP.

    Dipakai menggantikan st.metric karena st.metric tersusun ke bawah satu per satu di layar HP."""
    isi = "".join(f"<div class='fl-stat'><div class='k'>{k}</div><div class='v'>{v}</div></div>" for k, v in items)
    st.markdown(f"<div class='fl-stats' style='--n:{n};--m:{m or n}'>{isi}</div>", unsafe_allow_html=True)


def tampil(fig):
    """Tampilkan grafik Plotly yang ramah layar sentuh: tanpa toolbar, tanpa zoom saat halaman digeser jari."""
    fig.update_layout(dragmode=False)
    fig.update_xaxes(fixedrange=True)
    fig.update_yaxes(fixedrange=True)
    st.plotly_chart(fig, width="stretch", config=PLOT_CFG)


def tell(ok, msg_ok):
    """Beri tahu hasil perintah dengan bahasa sederhana (notifikasi kecil di pojok)."""
    if ok:
        st.toast(msg_ok, icon=":material/check_circle:")
    else:
        st.toast("Perintah belum terkirim. Pastikan lampu terhubung (lihat status di kanan atas).", icon=":material/error:")


def koneksi():
    """Status koneksi dalam bahasa awam: (teks, warna latar, warna teks)."""
    s = hub.status_now()
    upd = s["updated"]
    baru = upd is not None and (datetime.now(fl.WIB) - pd.Timestamp(upd).to_pydatetime()) < timedelta(seconds=20)
    if hub.mode_simulasi:
        return "Memakai data contoh", "#EDE7F6", "#4527A0"
    if not hub.conn["connected"]:
        return "Server belum terhubung", "#FDE8E8", "#B42318"
    if not baru:
        return "Lampu belum tersambung", "#FFF4DB", "#8A5A00"
    return "Lampu terhubung", "#E3F5EC", "#1E7A4C"


def kondisi(s):
    """Terjemahkan state alat menjadi judul, kalimat, warna cincin, dan petunjuk untuk pengguna awam."""
    state, rem, tgt = s["state"], s["remaining_s"], s["target_s"]
    menit = f"{int(rem) // 60} menit" if rem and rem >= 60 else f"{int(rem or 0)} detik"
    if state in ("FOCUS", "AWAY") and s["dock"] is False:
        return ("HP diangkat", "Kembalikan HP ke dudukan dalam beberapa detik, kalau tidak sesi akan berhenti.",
                "#E5484D", "Lampu berkedip merah dan lampu kecil DOCK ikut berkedip.")
    table = {
        "IDLE": ("Letakkan HP di dudukan", "Lampu menyala setelah HP terpasang di dudukannya.", "#8A94A8",
                 "Di simulasi Wokwi: geser slide switch ke posisi dock."),
        "READY": ("Siap belajar", "Pilih lama belajar di bawah, lalu tekan Mulai fokus. Bisa juga dengan tombol hijau di lampu.",
                  "#6FA8FF", "Lampu menyala biru redup dan lampu kecil DOCK menyala hijau."),
        "FOCUS": ("Kamu sedang fokus", f"Sisa <span id='sisa'>{menit}</span>. Tetap di meja dan biarkan HP di dudukan.", LAMP,
                  "Lampu putih; kecerahannya menyesuaikan cahaya ruangan."),
        "AWAY": ("Kamu meninggalkan meja",
                 "Timer dijeda. Kembali ke meja sebelum sesi dianggap ditinggal." if s["paused"]
                 else "Kembali ke meja agar sesi berlanjut.", "#F28C28", "Lampu menyala kuning."),
        "SELESAI": ("Sesi tuntas", "Kerja bagus! Istirahat sebentar sebelum sesi berikutnya.", "#2FBF71", "Lampu hijau."),
        "END_PAKSA": ("Sesi dihentikan", "Sesi berhenti sebelum waktunya dan tetap tercatat di riwayat.", "#E5484D", "Lampu merah."),
        "DITINGGAL": ("Sesi ditinggal", "Kamu terlalu lama meninggalkan meja, jadi sesi diakhiri.", "#B5654A", "Lampu merah redup."),
    }
    return table.get(state, ("Menunggu data dari lampu", "Nyalakan simulasi Wokwi. Status akan muncul di sini dalam beberapa detik.",
                             "#8A94A8", "Jika lama tidak muncul, buka menu Bantuan."))


HERO_HTML = """<!doctype html><html><head><meta charset="utf-8">
<link href="https://fonts.googleapis.com/css2?family=Plus+Jakarta+Sans:wght@400;500;700;800&display=swap" rel="stylesheet">
<style>
  html, body { margin:0; background:transparent; font-family:'Plus Jakarta Sans', system-ui, sans-serif; }
  .hero { display:flex; align-items:center; gap:34px; background:#22304F; color:#fff; border-radius:28px;
          padding:28px 34px; box-sizing:border-box; min-height:262px; }
  .ring { width:200px; height:200px; flex:0 0 200px; border-radius:50%; display:grid; place-items:center;
          background: conic-gradient(var(--c) calc(var(--p) * 1%), rgba(255,255,255,0.13) 0);
          box-shadow: 0 0 46px var(--glow); }
  .in { width:158px; height:158px; border-radius:50%; background:#22304F; display:grid; place-items:center; text-align:center; }
  .t { font-size:2.6rem; font-weight:800; font-variant-numeric:tabular-nums; line-height:1; }
  .l { font-size:0.82rem; color:#C9D1E3; margin-top:6px; }
  .st { font-size:2rem; font-weight:800; line-height:1.15; margin-bottom:8px; }
  .say { font-size:1.08rem; color:#DCE2EE; line-height:1.55; max-width:52ch; }
  .hint { margin-top:14px; font-size:0.92rem; color:#AEB8CE; }
  /* Layar HP: cincin besar di tengah dan teks di bawahnya, seperti layar timer aplikasi */
  @media (max-width: 600px) {
    .hero { flex-direction:column; text-align:center; gap:16px; padding:24px 20px 22px; border-radius:24px; min-height:0; }
    .ring { width:190px; height:190px; flex-basis:auto; }
    .in { width:150px; height:150px; }
    .t { font-size:2.5rem; } .l { font-size:0.8rem; }
    .st { font-size:1.4rem; margin-bottom:6px; } .say { font-size:0.95rem; line-height:1.5; margin:0 auto; }
    .hint { font-size:0.82rem; margin-top:10px; }
  }
</style></head><body>
<div class="hero">
  <div class="ring" id="ring" style="--p:__P__; --c:__C__; --glow:__G__">
    <div class="in"><div><div class="t" id="t">__T__</div><div class="l">__L__</div></div></div>
  </div>
  <div><div class="st">__J__</div><div class="say">__K__</div><div class="hint">__H__</div></div>
</div>
<script>
  // Hitung mundur berjalan di browser setiap 0,2 detik. Kecepatan (RATE) diukur dari data alat,
  // jadi tetap sama dengan OLED Wokwi walaupun simulasi berjalan lebih lambat dari waktu nyata.
  const REM = __REM__, AT = __AT__, RATE = __RATE__, RUN = __RUN__, TGT = __TGT__, SERVER_NOW = __NOW__;
  const offset = Date.now() / 1000 - SERVER_NOW;      // selisih jam browser dan jam server
  const fmt = s => { s = Math.max(0, Math.ceil(s)); return String(Math.floor(s / 60)).padStart(2, "0") + ":" + String(s % 60).padStart(2, "0"); };
  function tick() {
    if (REM === null) return;
    const now = Date.now() / 1000 - offset;
    const r = RUN ? Math.max(0, REM - RATE * (now - AT)) : REM;
    document.getElementById("t").textContent = fmt(r);
    const m = document.getElementById("sisa");
    if (m) m.textContent = r >= 60 ? Math.ceil(r / 60) + " menit" : Math.ceil(r) + " detik";
    if (TGT > 0 && __PROGRESS__) document.getElementById("ring").style.setProperty("--p", Math.min(100, 100 * r / TGT).toFixed(2));
  }
  tick();
  if (RUN) setInterval(tick, 200);

  // Tinggi kartu mengikuti isinya: di HP kartu lebih tinggi karena cincin dan teks disusun ke bawah.
  // Bingkai iframe dan wadah elemen Streamlit di luarnya ikut diubah tingginya.
  function fit() {
    try {
      const f = window.frameElement, h = Math.ceil(document.querySelector(".hero").getBoundingClientRect().height) + 4;
      if (!f) return;
      f.style.height = h + "px"; f.setAttribute("height", h);
      const box = f.closest('[data-testid="stElementContainer"]');
      if (box) box.style.height = h + "px";
    } catch (e) {}
  }
  fit(); window.addEventListener("resize", fit);
  if (window.ResizeObserver) new ResizeObserver(fit).observe(document.querySelector(".hero"));
</script></body></html>"""


def hero():
    """Kartu utama: cincin hitung mundur (meniru cincin lampu) + kalimat status + petunjuk.

    Hitung mundurnya dijalankan di browser agar mulus. Server hanya mengirim titik acuan (sisa detik,
    waktu diterima, kecepatan timer alat). Titik acuan lama dipakai terus selama prediksinya masih cocok
    dengan data baru, supaya kartu tidak dimuat ulang setiap 3 detik."""
    s = hub.status_now()
    judul, kalimat, warna, petunjuk = kondisi(s)
    cd = hub.countdown()
    lama = st.session_state.get("_acuan")
    if lama and cd["rem"] is not None and lama["running"] == cd["running"] and lama["state"] == cd["state"] \
            and lama["target"] == cd["target"] and lama["rem"] is not None:
        prediksi = lama["rem"] - (lama["rate"] * (cd["at"] - lama["at"]) if lama["running"] else 0)
        if abs(prediksi - cd["rem"]) <= 2.5 and abs(lama["rate"] - cd["rate"]) <= 0.15:
            cd = lama                                   # data baru masih sesuai prediksi: kartu tidak perlu dimuat ulang
    if cd is not lama:
        cd = dict(cd, now=time.time())
        st.session_state["_acuan"] = cd
    rem, tgt = cd["rem"], cd["target"]
    persen = 100 if s["state"] == "SELESAI" else (0 if not tgt else max(0, min(100, 100 * (rem or 0) / tgt)))
    glow = warna + "66" if s["state"] in ("FOCUS", "READY", "SELESAI") else "transparent"
    ganti = {"__P__": f"{persen:.2f}", "__C__": warna, "__G__": glow,
             "__T__": fl.fmt_mmss(rem) if rem is not None else "--:--",
             "__L__": "sisa waktu" if s["state"] in ("FOCUS", "AWAY") else "lama sesi",
             "__J__": judul, "__K__": kalimat, "__H__": petunjuk,
             "__REM__": "null" if rem is None else f"{rem:.2f}", "__AT__": f"{cd['at']:.3f}",
             "__RATE__": f"{cd['rate']:.2f}", "__RUN__": "true" if cd["running"] else "false",
             "__TGT__": str(int(tgt or 0)), "__NOW__": f"{cd['now']:.3f}",
             "__PROGRESS__": "false" if s["state"] == "SELESAI" else "true"}
    html = HERO_HTML
    for k, v in ganti.items():
        html = html.replace(k, v)
    components.html(html, height=266)
    return s


def ringkasan_hari_ini():
    """Tiga angka sederhana untuk hari ini: jumlah sesi, menit fokus, sesi tuntas."""
    df = hub.sessions_df()
    if not df.empty:
        tgl = pd.to_datetime(df["start_ts"], unit="s", utc=True).dt.tz_convert("Asia/Jakarta").dt.date
        df = df[tgl == datetime.now(fl.WIB).date()]
    st.markdown("<div class='fl-label'>Hari ini</div>", unsafe_allow_html=True)
    angka([("Sesi", len(df)),
           ("Menit fokus", f"{(df['focus_s'].sum() / 60 if len(df) else 0):.0f}"),
           ("Sesi tuntas", int((df["result"] == "SELESAI").sum()) if len(df) else 0)], n=3)


# ---------------------------------------------------------------------
# Kepala halaman + status koneksi
# ---------------------------------------------------------------------
@st.fragment(run_every=3)
def kepala():
    """Nama aplikasi dan status koneksi lampu (diperbarui otomatis)."""
    teks, bg, fg = koneksi()
    st.markdown(f"""<div class="fl-top"><div class="fl-brand">FocusLamp<small>Teman belajar tanpa gangguan HP</small></div>
<span class="fl-pill" style="background:{bg};color:{fg}"><span class="fl-pdot" style="background:{fg}"></span>{teks}</span></div>""", unsafe_allow_html=True)


kepala()
tab_home, tab_hist, tab_ai, tab_help = st.tabs([":material/home: Beranda", ":material/calendar_month: Riwayat", ":material/lightbulb: Saran AI", ":material/help: Bantuan"])


# ---------------------------------------------------------------------
# Tab Beranda
# ---------------------------------------------------------------------
@st.fragment(run_every=3)
def beranda_status():
    """Kartu status besar dan angka hari ini; diperbarui otomatis tiap 3 detik."""
    s = hero()
    if s["state"] in ("FOCUS", "AWAY"):
        if st.button("Hentikan sesi", help="Sesi berhenti sekarang dan tercatat sebagai dihentikan."):
            tell(*hub.stop_session()[:1], "Sesi dihentikan")
    st.session_state["_state_alat"] = s["state"]


@st.fragment(run_every=10)
def hari_ini():
    """Angka hari ini dan saran AI singkat (kolom kanan di laptop, di bawah tombol mulai di HP)."""
    ringkasan_hari_ini()
    res = fl.hitung_rekomendasi(hub.sessions_df(), pakai_demo=st.session_state.get("pakai_demo", False))
    isi = f"Coba {res['durasi']} menit per sesi" if res["ok"] else "Muncul setelah beberapa sesi"
    st.markdown(f"<div class='fl-mini'><div class='k'>Saran AI minggu ini</div><div class='v'>{isi}</div>"
                f"<div class='k' style='margin-top:4px'>Alasannya ada di menu Saran AI.</div></div>", unsafe_allow_html=True)


def kirim_durasi(mode, uji):
    """Dipanggil saat pilihan durasi diubah: langsung kirim set_duration ke alat supaya OLED Wokwi
    menampilkan durasi baru. Hitung mundur baru berjalan setelah tombol Mulai fokus ditekan.
    Tidak dikirim saat sesi berjalan (alat menolak), saat web belum terhubung, atau saat memakai data contoh."""
    pilihan = st.session_state.get(f"durasi_{mode}")
    if pilihan == "atur":
        durasi = st.session_state.get(f"durasi_sendiri_{mode}", 4.0 if uji else 40.0)
    else:
        durasi = pilihan
    if durasi is None or not hub.conn["connected"] or hub.mode_simulasi:
        return
    if hub.status_now()["state"] in ("FOCUS", "AWAY"):
        st.toast("Sesi sedang berjalan. Durasi baru berlaku untuk sesi berikutnya.", icon=":material/info:")
        return
    tell(hub.set_duration(float(durasi))[0], f"Lampu diatur ke {float(durasi):g} menit")


@st.fragment(run_every=3)
def mulai_sesi():
    """Pilihan lama belajar dan tombol Mulai fokus; status tombol ikut diperbarui otomatis."""
    st.subheader("Mulai sesi belajar")
    s = hub.status_now()
    state_alat, uji = s["state"], bool(s["demo"])
    siap = state_alat == "READY" and hub.conn["connected"] and not hub.mode_simulasi
    mode = "uji" if uji else "normal"
    opsi, bawaan = (PRESETS_UJI, 2) if uji else (PRESETS, 25)
    # Pilihan cepat + "Atur sendiri" untuk durasi bebas (0,5-180 menit, batas yang diterima ESP32)
    pilihan = st.segmented_control("Mau belajar berapa lama?", options=opsi + ["atur"], default=bawaan,
                                   format_func=lambda m: "Atur sendiri" if m == "atur" else f"{m} menit",
                                   key=f"durasi_{mode}", on_change=kirim_durasi, args=(mode, uji))
    if pilihan == "atur":
        durasi = st.number_input("Lama belajar (menit)", min_value=0.5 if uji else 1.0, max_value=180.0,
                                 value=4.0 if uji else 40.0, step=0.5 if uji else 1.0, format="%g",
                                 key=f"durasi_sendiri_{mode}", on_change=kirim_durasi, args=(mode, uji),
                                 help=f"Ketik angka atau pakai tombol - / +. Batas {'0,5' if uji else '1'} sampai 180 menit.")
    else:
        durasi = pilihan or bawaan
    if not uji and state_alat == "READY":
        selesai = datetime.now(fl.WIB) + timedelta(minutes=float(durasi))
        st.caption(f"Kalau mulai sekarang, sesi selesai sekitar pukul {selesai:%H:%M}.")
    # Durasi sudah dikirim saat dipilih; Mulai fokus mengirimnya sekali lagi agar pasti sama, lalu memulai sesi.
    if st.button(f"Mulai fokus {durasi:g} menit", type="primary", disabled=not siap):
        ok1, _ = hub.set_duration(durasi)
        ok2, _ = hub.start_session()
        tell(ok1 and ok2, f"Sesi {durasi:g} menit dimulai")
    if uji:
        st.caption("Lampu sedang di mode uji coba, jadi pilihan durasinya pendek untuk pengujian dan demo.")
    if not siap:
        if hub.mode_simulasi:
            st.caption("Sedang memakai data contoh. Kembali ke data asli di tab Bantuan untuk mengontrol lampu.")
        elif not hub.conn["connected"]:
            st.caption("Tombol aktif setelah web terhubung ke server.")
        elif state_alat in ("FOCUS", "AWAY"):
            st.caption("Sesi sedang berjalan. Sesi baru bisa dimulai setelah sesi ini selesai.")
        else:
            st.caption("Letakkan HP di dudukan dulu. Tombol aktif saat status menunjukkan Siap belajar.")


with tab_home:
    beranda_status()
    # Laptop: dua kolom (mulai sesi | ringkasan). HP: Streamlit otomatis menyusunnya ke bawah.
    kiri, kanan = st.columns([3, 2], gap="large")
    with kanan:
        hari_ini()
    with kiri:
        with st.container(border=True):
            mulai_sesi()
        with st.expander("Pengaturan lanjutan"):
            demo_now = bool(hub.status_now()["demo"])
            mode = st.radio("Mode waktu", ["Normal (untuk dipakai sehari-hari)", "Uji coba (deteksi dipercepat untuk demo)"],
                            index=1 if demo_now else 0)
            if st.button("Simpan mode", disabled=not (hub.conn["connected"] and not hub.mode_simulasi)):
                tell(hub.set_demo(mode.startswith("Uji"))[0], "Mode disimpan")
            st.caption("Saran AI hanya memakai sesi mode Normal. Mode hanya bisa diganti saat tidak ada sesi berjalan.")

    with st.expander("Detail sensor (untuk teknisi dan juri)"):
        @st.fragment(run_every=3)
        def detail_sensor():
            """Grafik jarak, cahaya, riwayat sesi, dan event terbaru seperti dashboard teknis."""
            s = hub.status_now()
            yn = lambda v: "-" if v is None else ("Ya" if v else "Tidak")
            angka([("State", s["state"]), ("Pengguna hadir", yn(s["present"])), ("HP di dudukan", yn(s["dock"])),
                   ("Cahaya ruangan", "-" if s["light_pct"] is None else f"{int(s['light_pct'])}%"),
                   ("Mode", "-" if s["demo"] is None else ("Uji coba" if s["demo"] else "Normal"))], n=5, m=2)
            tel, ev, _, _ = hub.frames()
            tampil(fl.fig_distance(tel))
            a, b = st.columns(2)
            with a:
                tampil(fl.fig_light(tel))
            with b:
                tampil(fl.fig_history(hub.sessions_df()))
            st.dataframe(fl.events_table(ev), width="stretch", hide_index=True)
        detail_sensor()


# ---------------------------------------------------------------------
# Tab Riwayat
# ---------------------------------------------------------------------
def tabel_ramah(df):
    """Tabel sesi dengan kolom berbahasa sehari-hari, terbaru di atas."""
    t = pd.to_datetime(df["start_ts"], unit="s", utc=True).dt.tz_convert("Asia/Jakarta")
    out = pd.DataFrame({
        "Tanggal": t.dt.strftime("%d %b %Y"), "Mulai": t.dt.strftime("%H:%M"),
        "Rencana (menit)": (df["target_s"] / 60).round(1), "Fokus (menit)": (df["focus_s"] / 60).round(1),
        "Hasil": df["result"].map(HASIL).fillna(df["result"]),
        "Penyebab berhenti": df["end_reason"].map(PENYEBAB).fillna(df["end_reason"]),
        "Pergi dari meja": df["away_count"].astype(int), "HP diangkat": df["dock_lift_count"].astype(int)})
    return out.iloc[::-1].reset_index(drop=True)


def grafik_harian(df, hari=14):
    """Grafik batang menit fokus per hari selama `hari` terakhir."""
    t = pd.to_datetime(df["start_ts"], unit="s", utc=True).dt.tz_convert("Asia/Jakarta").dt.date
    per = (df.assign(tgl=t).groupby("tgl")["focus_s"].sum() / 60)
    akhir = datetime.now(fl.WIB).date()
    idx = [akhir - timedelta(days=i) for i in range(hari - 1, -1, -1)]
    per = per.reindex(idx, fill_value=0)
    label = [tgl_id(d) for d in per.index]
    fig = go.Figure(go.Bar(x=label, y=per.values, marker_color=LAMP,
                           hovertemplate="%{x}: %{y:.0f} menit fokus<extra></extra>"))
    fig.update_layout(height=250, template="plotly_white", margin=dict(l=4, r=4, t=10, b=4),
                      yaxis_title="menit", font=dict(family="Plus Jakarta Sans, sans-serif"), bargap=0.25)
    # label tanggal ditulis tiap 2 hari dan tidak dimiringkan, supaya tetap terbaca di layar HP
    fig.update_xaxes(tickangle=0, tickmode="array", tickvals=label[1::2], ticktext=label[1::2])
    return fig


def daftar_sesi(df, n=8):
    """Sesi terbaru sebagai kartu: tanggal, hasil (berwarna), menit fokus dari rencana, penyebab berhenti.

    Kartu dipakai alih-alih tabel lebar karena tabel harus digeser ke samping di layar HP."""
    t = pd.to_datetime(df["start_ts"], unit="s", utc=True).dt.tz_convert("Asia/Jakarta")
    out = []
    for (_, r), w in list(zip(df.iterrows(), t))[::-1][:n]:
        hasil = HASIL.get(r["result"], r["result"])
        bg, fg = WARNA_HASIL.get(hasil, ("#E6EBF4", "#22304F"))
        fokus, rencana = r["focus_s"] / 60, r["target_s"] / 60
        persen = 0 if not rencana else max(0, min(100, 100 * fokus / rencana))
        alasan = PENYEBAB.get(r["end_reason"], r["end_reason"])
        out.append(f"""<div class='fl-sesi'><div class='top'><div class='d'>{tgl_id(w)} · {w:%H:%M}</div>
<span class='fl-badge' style='background:{bg};color:{fg}'>{hasil}</span></div>
<div class='r'>Fokus {fokus:.0f} dari {rencana:g} menit · {alasan}</div>
<div class='bar'><span style='width:{persen:.0f}%;background:{fg}'></span></div></div>""")
    st.markdown("".join(out), unsafe_allow_html=True)


with tab_hist:
    df_all = hub.sessions_df()
    if df_all.empty:
        st.info("Belum ada sesi. Setelah kamu menyelesaikan sesi pertama, riwayatnya muncul di sini.")
    else:
        t = pd.to_datetime(df_all["start_ts"], unit="s", utc=True).dt.tz_convert("Asia/Jakarta")
        minggu = df_all[t >= pd.Timestamp(datetime.now(fl.WIB) - timedelta(days=7))]
        st.subheader("7 hari terakhir")
        jam_terbaik = "-"
        if len(minggu):
            jam = pd.to_datetime(minggu["start_ts"], unit="s", utc=True).dt.tz_convert("Asia/Jakarta").dt.hour
            terbaik = minggu.assign(jam=jam).groupby("jam")["result"].apply(lambda r: (r == "SELESAI").mean()).idxmax()
            jam_terbaik = f"{int(terbaik):02d}:00"
        angka([("Sesi", len(minggu)),
               ("Total fokus", f"{minggu['focus_s'].sum() / 60:.0f} <small>menit</small>"),
               ("Sesi tuntas", f"{(minggu['result'] == 'SELESAI').mean() * 100:.0f}%" if len(minggu) else "-"),
               ("Jam paling fokus", jam_terbaik)], n=4, m=2)
        # Laptop: grafik di kiri, daftar sesi di kanan. HP: tersusun ke bawah.
        kiri, kanan = st.columns([3, 2], gap="large")
        with kiri:
            with st.container(border=True):
                st.markdown("**Menit fokus per hari**")
                tampil(grafik_harian(df_all))
            with st.expander("Grafik lengkap (analitik)"):
                tampil(fl.fig_analytics(df_all))
        with kanan:
            st.markdown("**Sesi terakhir**")
            daftar_sesi(df_all, n=5)
        with st.expander("Tabel lengkap"):
            st.dataframe(tabel_ramah(df_all), width="stretch", hide_index=True)
        st.download_button("Unduh riwayat (CSV)", df_all.to_csv(index=False).encode("utf-8"),
                           file_name="sessions.csv", mime="text/csv")


# ---------------------------------------------------------------------
# Tab Saran AI
# ---------------------------------------------------------------------
with tab_ai:
    pakai_demo = st.session_state.get("pakai_demo", False)
    res = fl.hitung_rekomendasi(hub.sessions_df(), pakai_demo=pakai_demo)
    if not res["ok"]:
        st.markdown(f"""<div class="fl-advice"><div class="big">Saran akan muncul setelah beberapa sesi</div>
<div class="sub">{res['pesan']} Saran dihitung dari kebiasaanmu sendiri, jadi makin sering dipakai makin tepat.</div></div>""",
                    unsafe_allow_html=True)
    else:
        d, T = res["durasi"], res["durasi_sekarang"]
        if d > T:
            sub = f"Kamu konsisten menuntaskan sesi {T} menit. Saatnya naik sedikit."
        elif d < T:
            sub = f"Sesi {T} menit terasa berat akhir-akhir ini. Turunkan dulu, nanti naik lagi saat sudah lancar."
        else:
            sub = f"Pertahankan {T} menit sampai sebagian besar sesi tuntas."
        st.markdown(f"""<div class="fl-advice"><div class="big">Coba belajar {d} menit per sesi</div>
<div class="sub">{sub}</div></div>""", unsafe_allow_html=True)
        if st.button(f"Pakai {d} menit untuk sesi berikutnya", type="primary",
                     disabled=not (hub.conn["connected"] and not hub.mode_simulasi)):
            tell(hub.set_duration(d)[0], f"Lampu diatur ke {d} menit")
        st.markdown("**Kenapa saran ini?**")
        st.markdown("\n".join(f"- {p}" for p in res["poin"]))
        with st.expander("Cara AI menghitung (detail)"):
            st.markdown("Saran membandingkan minggu ini dengan minggu lalu, menghitung peluang kamu bertahan fokus "
                        "sampai menit tertentu (Kaplan–Meier berbobot waktu), dan mencari jam belajar terbaik "
                        "(Logistic Regression). Sesi terbaru diberi bobot lebih besar.")
            st.dataframe(res["tabel"], width="stretch")
            tampil(fl.fig_rekomendasi(res))
    st.toggle("Ikutkan sesi mode uji coba (untuk mencoba saja)", key="pakai_demo")


# ---------------------------------------------------------------------
# Tab Bantuan
# ---------------------------------------------------------------------
with tab_help:
    st.subheader("Cara pakai")
    st.markdown("""
1. **Letakkan HP di dudukan.** Lampu menyala biru redup, artinya siap.
2. **Pilih lama belajar** di Beranda (pilihan cepat, atau **Atur sendiri** untuk durasi bebas), lalu tekan **Mulai fokus**. Bisa juga dengan tombol hijau di lampu.
3. **Belajar sampai waktu habis.** Kalau kamu pergi atau mengangkat HP, lampu memberi tanda dan sesi bisa berhenti.
4. **Lihat Riwayat dan Saran AI** setiap minggu untuk tahu durasi belajar yang paling cocok.
""")
    st.subheader("Arti warna lampu")
    for warna, arti in [("#FFFFFF", "Putih: sedang fokus (makin gelap ruangan, makin terang lampunya)"),
                        ("#6FA8FF", "Biru redup: HP sudah di dudukan, siap mulai"),
                        ("#F28C28", "Kuning: kamu meninggalkan meja"),
                        ("#E5484D", "Merah berkedip: HP diangkat, atau sesi dihentikan"),
                        ("#2FBF71", "Hijau: sesi tuntas"),
                        ("#C9CED8", "Mati: HP belum di dudukan")]:
        st.markdown(f'<div class="fl-legend"><span class="fl-dot" style="background:{warna}"></span>{arti}</div>',
                    unsafe_allow_html=True)
    st.markdown("**Lampu kecil hijau (DOCK)** menyala saat HP terpasang di dudukan, dan berkedip bila HP diangkat ketika sesi berjalan.")
    st.markdown("**Lampu kecil kuning (NET)** menyala terus saat lampu tersambung ke server. Berkedip pelan berarti WiFi putus, "
                "berkedip cepat berarti WiFi ada tetapi server belum tersambung.")
    st.info("Kalau internet putus, lampu tetap bisa dipakai belajar seperti biasa. Hasil sesi disimpan di lampu dan "
            "otomatis masuk ke Riwayat setelah lampu tersambung lagi.", icon=":material/wifi_off:")

    st.subheader("Pasang di HP seperti aplikasi")
    st.markdown("""
Web ini bisa dibuka di laptop maupun HP. Di HP, tampilannya berubah seperti aplikasi dengan menu di bagian bawah.
1. Buka link web di **Chrome** (Android) atau **Safari** (iPhone).
2. Ketuk menu titik tiga lalu pilih **Tambahkan ke layar utama** atau **Instal aplikasi** (Safari: tombol Bagikan > *Add to Home Screen*).
3. Ikon **FocusLamp** muncul di layar utama dan terbuka layar penuh tanpa bilah alamat.

Catatan: link berganti setiap notebook Colab dijalankan ulang, jadi ikon perlu dipasang ulang setelah link baru dibuat.
""")

    st.subheader("Coba tanpa alat")
    st.caption("Isi web dengan data contoh untuk melihat tampilannya. Data contoh tidak tersimpan ke riwayat.")
    a, b, c = st.columns(3)
    if a.button("Contoh 3 sesi", width="stretch"):
        hub.simulate_demo(); st.rerun()
    if b.button("Contoh 2 minggu (untuk Saran AI)", width="stretch"):
        hub.simulate_history(); st.rerun()
    if c.button("Kembali ke data asli", width="stretch", disabled=not hub.mode_simulasi):
        n = hub.back_to_real(); st.toast(f"{n} sesi dimuat dari riwayat"); st.rerun()

    with st.expander("Info teknis"):
        st.markdown(f"""
- Perangkat: `{hub.device_id}`, topic `{hub.topic_base}/#`
- Server MQTT: `{hub.broker or '-'}` ({'tersambung' if hub.conn['connected'] else 'belum tersambung'})
- Pesan terakhir: {hub.conn['last_error'] or '-'}
- File riwayat: `{hub.history_path}` ({hub.history_count()} sesi tersimpan)
- Perintah yang dikirim web sama dengan aplikasi Android nanti: `set_duration`, `start`, `stop`, `set_mode`.
""")

In [ ]:
# Ikon dan manifest agar web bisa dipasang di layar utama HP (disajikan Streamlit di alamat app/static/)
import json, os
from PIL import Image, ImageDraw, ImageFilter
os.makedirs("static", exist_ok=True)

def buat_ikon(n, maskable=False):
    S = n * 4
    im = Image.new("RGBA", (S, S), (0, 0, 0, 0)); d = ImageDraw.Draw(im)
    if maskable:
        d.rectangle([0, 0, S, S], fill="#22304F")
    else:
        d.rounded_rectangle([0, 0, S - 1, S - 1], radius=int(S * 0.22), fill="#22304F")
    c, r, w = S / 2, S * (0.27 if maskable else 0.32), S * 0.085
    glow = Image.new("RGBA", (S, S), (0, 0, 0, 0))
    ImageDraw.Draw(glow).ellipse([c - r - w, c - r - w, c + r + w, c + r + w], outline=(242, 179, 61, 150), width=int(w * 1.6))
    im.alpha_composite(glow.filter(ImageFilter.GaussianBlur(S * 0.03)))
    d = ImageDraw.Draw(im)
    d.ellipse([c - r, c - r, c + r, c + r], outline=(255, 255, 255, 40), width=int(w))
    d.arc([c - r, c - r, c + r, c + r], start=-90, end=200, fill="#F2B33D", width=int(w))
    rr = S * (0.075 if maskable else 0.09)
    d.ellipse([c - rr, c - rr, c + rr, c + rr], fill="#FFF3D6")
    return im.resize((n, n), Image.LANCZOS)

buat_ikon(192).save("static/icon-192.png"); buat_ikon(512).save("static/icon-512.png")
buat_ikon(512, True).save("static/icon-maskable-512.png")
json.dump({"name": "FocusLamp", "short_name": "FocusLamp", "description": "Teman belajar tanpa gangguan HP",
           "start_url": "../../", "scope": "../../", "display": "standalone", "orientation": "portrait",
           "background_color": "#F7F8FB", "theme_color": "#22304F", "lang": "id",
           "icons": [{"src": "icon-192.png", "sizes": "192x192", "type": "image/png"},
                     {"src": "icon-512.png", "sizes": "512x512", "type": "image/png"},
                     {"src": "icon-maskable-512.png", "sizes": "512x512", "type": "image/png", "purpose": "maskable"}]},
          open("static/manifest.json", "w"), indent=2)
print("Ikon dan manifest siap di folder static/")

## 5. Jalankan dashboard
Sel ini menyalakan Streamlit di belakang layar, membuat link publik lewat cloudflared, lalu menampilkan link-nya. Jalankan ulang sel ini kapan pun ingin me-restart web (misalnya setelah mengubah sel 2).

- **Buka link sekali** setelah muncul: koneksi ke EMQX baru dimulai saat halaman pertama kali dibuka.
- Link berganti setiap sel ini dijalankan. Link bisa dibuka siapa pun yang memilikinya, jadi bagikan hanya ke tim dan juri.
- Kalau link utama tidak muncul, pakai **link cadangan** (hanya bisa dibuka di browser yang sedang membuka Colab ini).

In [ ]:
import re, subprocess, sys, time, urllib.request
from IPython.display import HTML, display

def stop_dashboard():
    for p in globals().get("_procs", []):
        try:
            p.terminate(); p.wait(timeout=5)
        except Exception:
            pass
    globals()["_procs"] = []

stop_dashboard()
env = dict(os.environ, FL_BROKER=BROKER, FL_PORT=str(PORT), FL_USERNAME=USERNAME, FL_PASSWORD=PASSWORD or "",
           FL_DEVICE_ID=DEVICE_ID, FL_HISTORY_PATH=HISTORY_PATH)
p_st = subprocess.Popen([sys.executable, "-m", "streamlit", "run", "app.py", "--server.port", "8501",
                         "--server.headless", "true", "--browser.gatherUsageStats", "false",
                         "--theme.base", "light", "--theme.primaryColor", "#22304F",
                         "--theme.backgroundColor", "#F7F8FB", "--theme.secondaryBackgroundColor", "#E6EBF4",
                         "--theme.textColor", "#1F2638",
                         "--server.enableCORS", "false", "--server.enableXsrfProtection", "false",
                         "--server.enableStaticServing", "true"],
                        env=env, stdout=open("streamlit.log", "w"), stderr=subprocess.STDOUT)
_procs = [p_st]

siap = False
for _ in range(60):                                   # tunggu Streamlit siap (maks. 60 detik)
    try:
        siap = urllib.request.urlopen("http://localhost:8501/_stcore/health", timeout=2).read() == b"ok"
    except Exception:
        pass
    if siap:
        break
    time.sleep(1)
print("Streamlit:", "berjalan" if siap else "belum siap, cek sel 6")

url = None
try:
    p_cf = subprocess.Popen(["./cloudflared", "tunnel", "--url", "http://localhost:8501", "--no-autoupdate"],
                            stdout=open("cloudflared.log", "w"), stderr=subprocess.STDOUT)
    _procs.append(p_cf)
    for _ in range(45):                               # tunggu link cloudflared (maks. 45 detik)
        m = re.search(r"https://(?!api\.)[-a-z0-9]+\.trycloudflare\.com", open("cloudflared.log").read())
        if m:
            url = m.group(0); break
        if p_cf.poll() is not None:                   # cloudflared berhenti (misalnya gagal membuat tunnel)
            break
        time.sleep(1)
    if url:                                           # tunggu tunnel benar-benar tersambung ke server Cloudflare
        for _ in range(30):
            if "Registered tunnel connection" in open("cloudflared.log").read():
                break
            time.sleep(1)
        print("Menunggu link aktif (DNS)...")
        bisa = False
        for _ in range(12):                           # cek dari Colab apakah link sudah bisa dibuka
            try:
                bisa = urllib.request.urlopen(url + "/_stcore/health", timeout=5).read() == b"ok"
            except Exception:
                pass
            if bisa:
                break
            time.sleep(5)
        print("Cek link dari Colab:", "BISA dibuka" if bisa else "belum bisa dibuka (lihat sel 6)")
except FileNotFoundError:
    print("cloudflared belum ada, jalankan sel 1 dulu.")

if url:
    display(HTML(f'<p style="font-size:18px">Dashboard FocusLamp: <a href="{url}" target="_blank">{url}</a></p>'
                 '<p>Buka link ini sekarang (wajib sekali) agar web mulai tersambung ke EMQX. '
                 'Kalau browser menulis "situs tidak dapat dijangkau", tunggu 30 detik lalu muat ulang. '
                 'Kalau tetap tidak bisa padahal cek dari Colab BISA, jaringanmu memblokir trycloudflare.com: '
                 'coba buka dari data seluler HP, atau pakai link cadangan di bawah.</p>'))
else:
    print("Link cloudflared belum muncul. Pakai link cadangan di bawah atau cek sel 6.")
try:
    from google.colab.output import eval_js
    print("Link cadangan (hanya di browser ini):", eval_js("google.colab.kernel.proxyPort(8501)"))
except Exception:
    pass

## 6. Cek log (kalau ada masalah)
Menampilkan 30 baris terakhir log Streamlit dan cloudflared. Status koneksi MQTT juga terlihat di pojok kanan atas web dan di menu Bantuan → Info teknis.

In [ ]:
for f in ("streamlit.log", "cloudflared.log"):
    print(f"===== {f} =====")
    try:
        print("".join(open(f).readlines()[-30:]))
    except FileNotFoundError:
        print("(belum ada)")

## 7. Matikan dashboard (opsional)
Menghentikan web dan link. Riwayat di Drive tidak terhapus.

In [ ]:
stop_dashboard()
print("Dashboard dimatikan.")